## RAGAS Evaluation — End-to-End Pipeline

Đánh giá chất lượng sinh câu trả lời của pipeline trên nhánh HEALTH_ADVICE.

**Dataset:** `data/en/eval_200.jsonl` — 200 Q&A pairs (LLM-generated ground truth)  
**Eval subset:** 30 câu HEALTH_ADVICE (filter bỏ abstract-style và NUTRITION_LOOKUP)  
**Judge LLM:** Ollama llama3.1:8b (local)  
**Embeddings:** sentence-transformers/all-MiniLM-L6-v2 (cho AnswerRelevancy)

**Metrics:**
| Metric | Mô tả |
|---|---|
| Faithfulness | Answer có bịa nội dung ngoài context không? |
| Answer Relevancy | Answer có trả lời đúng câu hỏi không? |
| Context Precision | Chunks retrieved có liên quan không? |
| Context Recall | Context có bao phủ ground truth không? |

> **Yêu cầu:** `ollama serve` đang chạy với model `llama3.1:8b` đã pull.

In [ ]:
import os, sys

if not os.path.exists('configs/config.yaml'):
    os.chdir('../..')
sys.path.insert(0, '.')

import json
import re
import yaml
import pandas as pd
from tqdm import tqdm

cfg = yaml.safe_load(open('configs/config.yaml'))
print(f'CWD: {os.getcwd()}')
print(f'LLM: {cfg["llm_model"]}')

In [ ]:
# Load eval_200.jsonl, filter to natural Q&A questions
# Skip NFCorpus abstract-style entries (question = full abstract, answer = title)

eval_data = []
with open('data/en/eval_200.jsonl', encoding='utf-8') as f:
    for line in f:
        eval_data.append(json.loads(line))

QA_RE = re.compile(
    r'^(what|how|why|which|can|does|do|is|are|should|will)\b',
    re.IGNORECASE
)

candidates = [
    d for d in eval_data
    if len(d['question']) < 350 and QA_RE.match(d['question'].strip())
]

EVAL_SIZE = 30
subset = candidates[:EVAL_SIZE]

print(f'Total eval_200   : {len(eval_data)}')
print(f'Q&A candidates   : {len(candidates)}')
print(f'Eval subset      : {len(subset)}')
print(f'Sample question  : {subset[0]["question"]}')

In [ ]:
# Init pipeline — models load lazily on first call
from src.en.pipeline import ENPipeline

pipeline = ENPipeline()
print('Pipeline ready')

In [ ]:
# Collect pipeline outputs + retrieved contexts for each question
# RAGAS only applies to HEALTH_ADVICE path (LLM + retrieval)
# NUTRITION_LOOKUP bypasses LLM entirely — skip those entries

records = []
skipped = 0

for entry in tqdm(subset, desc='Collecting'):
    question     = entry['question']
    ground_truth = entry['answer']

    result = pipeline.answer(question)
    intent = result['intent']

    if intent == 'NUTRITION_LOOKUP':
        skipped += 1
        continue

    # Re-run retrieval to capture chunk texts for RAGAS contexts
    raw_candidates = pipeline.retriever.retrieve(question, top_k=20)
    chunks         = pipeline.reranker.rerank(question, raw_candidates, top_k=pipeline.top_k)
    contexts       = [c.text for c in chunks]

    if not contexts:
        skipped += 1
        continue

    records.append({
        'question'     : question,
        'answer'       : result['answer'],
        'contexts'     : contexts,
        'ground_truth' : ground_truth,
    })

print(f'Collected : {len(records)}')
print(f'Skipped   : {skipped} (NUTRITION_LOOKUP or no context)')

In [ ]:
# Configure RAGAS with Ollama judge + local embeddings
# ragas==0.1.x API: metric.llm / metric.embeddings setters

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import HuggingFaceEmbeddings

judge_llm = LangchainLLMWrapper(
    ChatOllama(model=cfg['llm_model'], temperature=0)
)
judge_emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name=cfg['embedding_model'])
)

for metric in [faithfulness, answer_relevancy, context_precision, context_recall]:
    metric.llm = judge_llm
answer_relevancy.embeddings = judge_emb

ds = Dataset.from_list(records)
print(f'RAGAS dataset: {len(ds)} samples')

In [ ]:
# Run RAGAS evaluation — uses Ollama as judge, expect ~5-15 min for 30 samples

scores = evaluate(
    dataset=ds,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
)

print(scores)

In [ ]:
# Results table

score_map = {
    'Faithfulness'      : scores['faithfulness'],
    'Answer Relevancy'  : scores['answer_relevancy'],
    'Context Precision' : scores['context_precision'],
    'Context Recall'    : scores['context_recall'],
}

df = pd.DataFrame(
    list(score_map.items()),
    columns=['Metric', 'Score']
).set_index('Metric')

df['Score'] = df['Score'].round(4)
print(df.to_string())

# Save scores to JSON for report
import json as _json
with open('reports/en/ragas_scores.json', 'w') as f:
    _json.dump({k: round(float(v), 4) for k, v in score_map.items()}, f, indent=2)
print('Saved: reports/en/ragas_scores.json')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
df['Score'].plot(
    kind='bar', ax=ax,
    title='RAGAS Scores — Hybrid+Reranker Pipeline (30 HEALTH_ADVICE queries)',
    ylim=(0, 1), rot=20, color='steelblue'
)
ax.set_ylabel('Score')
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8)

for i, v in enumerate(df['Score']):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('reports/en/ragas_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/en/ragas_scores.png')